# Rung 2 — Dixon–Coles

We have a measured baseline (rung 1). Now we add the two Dixon–Coles upgrades and **only keep them if they beat that baseline on RPS**:

1. **Low-score correction** — independent Poissons under-count 0-0/1-0/0-1/1-1. DC multiplies those four cells by a factor governed by one parameter ρ.
2. **Time decay** — weight recent matches more via `exp(-ξ·age)`, replacing rung 1's hard year cutoff.

The model lives in `../src/dixon_coles.py`. The headline lesson of this notebook: **the win comes from the decay, and only at the right ξ — so you have to tune and measure, not assume.**

In [ ]:
import sys, os; sys.path.append(os.path.abspath(".."))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src import data, poisson, dixon_coles as dc, evaluate
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

## 1. Data — same split for a fair fight

Both models must see exactly the same train/test split, or the comparison is meaningless. We also apply `filter_teams` to drop micro-nations with tiny samples (faster fit, stabler strengths).

In [ ]:
CSV = "../data/raw/results.csv"
df = data.load_results(CSV) if os.path.exists(CSV) else data.make_synthetic(12, 800, 2)
df = data.filter_recent(df, years=8)
df = data.filter_teams(df, min_matches=25)
d = df.sort_values("date"); cut = int(len(d) * 0.8)
train, test = d.iloc[:cut], d.iloc[cut:]
print(f"{len(df):,} matches, {len(set(df.home_team)|set(df.away_team))} teams | train {len(train):,} / test {len(test):,}")

## 2. Backtest helper
Precompute the test fixtures once, then score any model's `[home,draw,away]` probabilities.

In [ ]:
pm = poisson.fit(train)
known = set(pm.teams)
pairs = [(r.home_team, r.away_team, evaluate.result_to_outcome(r.home_score, r.away_score))
         for r in test.itertuples() if r.home_team in known and r.away_team in known]

def backtest(predict):
    rows = [[(p := predict(h, a))["home"], p["draw"], p["away"]] for h, a, _ in pairs]
    return evaluate.mean_scores(rows, [o for *_, o in pairs])

baseline = backtest(lambda h, a: pm.outcome_probs(h, a))
print("Rung 1 (Poisson):", {k: round(v,4) if isinstance(v,float) else v for k,v in baseline.items()})

## 3. Tune the decay ξ

This is the experiment. Fit DC at several decay rates and read RPS off each. ξ=0 means "all history weighted equally" (so any gain over that is purely the decay doing work).

In [ ]:
grid = [0.0, 0.0005, 0.001, 0.0019, 0.003]
rows = []
for xi in grid:
    m = dc.fit(train, xi=xi)
    s = backtest(lambda h, a: m.outcome_probs(h, a, neutral=False))
    half_life = np.inf if xi == 0 else round(np.log(2)/xi/365, 2)
    rows.append({"xi": xi, "half_life_yrs": half_life, "rho": round(m.rho,3), "rps": round(s["rps"],4)})
sweep = pd.DataFrame(rows)
sweep["vs_baseline"] = round(baseline["rps"],4) - sweep["rps"]   # positive = better than rung 1
sweep

In [ ]:
best = sweep.loc[sweep.rps.idxmin()]
print(f"Best ξ = {best.xi} (half-life ≈ {best.half_life_yrs} yrs): RPS {best.rps:.4f}")
print(f"Rung-1 baseline:                          RPS {baseline['rps']:.4f}")
verdict = "DC WINS — keep it" if best.rps < baseline["rps"] else "no improvement — baseline stands"
print("→", verdict)

plt.figure(figsize=(6,3.5))
plt.axhline(baseline["rps"], ls="--", c="gray", label="Rung 1 (Poisson)")
plt.plot(sweep.xi, sweep.rps, "o-", label="Dixon–Coles")
plt.xlabel("ξ (time-decay rate)"); plt.ylabel("RPS (lower better)"); plt.legend(); plt.tight_layout(); plt.show()

## 4. What did ρ actually do?

ρ is small and negative — it nudges probability toward the low-score draws the plain model misses. Compare the two scoreline grids for an evenly-matched fixture: DC should fatten the 0-0 / 1-1 cells slightly.

In [ ]:
m = dc.fit(train, xi=float(best.xi))
h_team, a_team = pm.teams[0], pm.teams[1]
pm_grid = pm.score_matrix(h_team, a_team)[:5, :5]
dc_grid = m.score_matrix(h_team, a_team, neutral=False)[:5, :5]

fig, ax = plt.subplots(1, 3, figsize=(12, 3.4))
for k, (g, t) in enumerate([(pm_grid, "Poisson"), (dc_grid, "Dixon–Coles"), (dc_grid - pm_grid, "DC − Poisson")]):
    im = ax[k].imshow(g, origin="lower", cmap=("RdBu" if k==2 else "viridis"))
    ax[k].set_title(f"{t}\n{h_team} vs {a_team}"); ax[k].set_xlabel("away"); ax[k].set_ylabel("home")
    fig.colorbar(im, ax=ax[k])
plt.tight_layout(); plt.show()
print("DC rho:", round(m.rho,3), "| home_adv:", round(m.home_adv,3))

## 5. Neutral venues — the World Cup wrinkle

Almost every WC2026 game is at a neutral venue, so home advantage shouldn't apply. `expected_goals(..., neutral=True)` zeroes it. Compare a fixture both ways to see how much the home-advantage term is worth.

In [ ]:
for nb_flag in (False, True):
    o = m.outcome_probs(h_team, a_team, neutral=nb_flag)
    tag = "neutral (WC)" if nb_flag else "home/away"
    print(f"{tag:14} H/D/A = {o['home']:.2f}/{o['draw']:.2f}/{o['away']:.2f}")

## Takeaways → rung 3

- DC's value here is the **time-decay**, tuned; the ρ correction mainly sharpens *exact scores*, not 1X2.
- For WC predictions, fit on home/away history but **predict with `neutral=True`**.
- **Rung 3** adds strength priors (Elo / FIFA ranking). This matters because WC group-stage teams have few recent comparable games — a prior stabilises strengths the data alone estimates poorly. That's the next `src/` module.